# Multi-Level HMM Portfolio Allocation

Combines HAR semivariance with a hierarchical HMM and regime-specific portfolio optimization.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
HAR-SEMIVARIANCE + HIERARCHICAL HMM + CVaR OPTIMIZER STRATEGY
=============================================================

ARCHITECTURE:
------------
Two-level hierarchical approach combining micro and macro volatility signals:

LEVEL 1 - MACRO (HHMM): Market Regime Detection
  ├─ State 1 (Low Vol):     120% exposure → Max Sharpe
  ├─ State 2 (Medium Vol):  100% exposure → HRP
  ├─ State 3 (High Vol):     60% exposure → MST-HRP Clusters
  └─ State 4 (Extreme Vol):  20% exposure → CVaR + Safe Haven

LEVEL 2 - MICRO (HAR-Semivariance): Asset-Level Volatility
  ├─ Daily component (1-day realized semivariance)
  ├─ Weekly component (5-day realized semivariance)
  ├─ Monthly component (22-day realized semivariance)
  └─ Focus on downside volatility (negative returns only)

ANTI-LOOK-AHEAD GUARANTEE:
-------------------------
- Features at T use ONLY data up to T-1 (double-lagged)
- Signal generated at Close[T-1]
- Execution at Open[T]
- Portfolio valued at Close[T]
- Returns: Close[T]/Close[T-1] - 1
- Walk-forward with quarterly HHMM refitting

EXECUTION FLOW:
--------------
Day T-1 (Close):
  ├─ Calculate HAR-Semivariance features (using T-2 data)
  ├─ Detect HHMM regime (using T-1 features)
  ├─ Run CVaR optimization (using historical returns up to T-1)
  └─ Generate signals (position sizes based on Close[T-1] value)

Day T (Open):
  ├─ Execute signals at Open[T]
  └─ Update positions

Day T (Close):
  ├─ Value portfolio at Close[T]
  └─ Calculate returns: Close[T]/Close[T-1] - 1

Author: Production Trading System
Date: 2025
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
from scipy.optimize import minimize
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from typing import Tuple, Dict, List, Optional
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from hmmlearn import hmm as hmm_lib
from dataclasses import dataclass
import warnings

warnings.filterwarnings("ignore")


class StrategyConfig:
    """
    Centralized configuration for all strategy parameters.

    CRITICAL: All parameters are configurable here - no hardcoding.
    """

    END_DATE = "2024-12-31"
    START_DATE = "2019-12-31"
    INITIAL_CAPITAL = 100000.0
    BENCHMARK = "URTH"
    HAR_DAILY_WINDOW = 1
    HAR_WEEKLY_WINDOW = 5
    HAR_MONTHLY_WINDOW = 22
    HHMM_N_STATES = 3
    HHMM_N_ITER = 200
    HHMM_RANDOM_STATE = 42
    HHMM_COVARIANCE_TYPE = "diag"
    MIN_TRAINING_DAYS = 252
    REFIT_FREQUENCY = 63
    REBALANCE_FREQUENCY = 21
    LOOKBACK_WINDOW = 252
    REGIME_EXPOSURES = {0: 1.2, 1: 1.0, 2: 0.5}
    REGIME_OPTIMIZERS = {0: "MaxSharpe", 1: "HRP", 2: "CVaR_SafeHaven"}
    CVAR_CONFIDENCE = 0.95
    CVAR_LOOKBACK = 252
    MAX_POSITION_SIZE = 0.2
    MIN_POSITION_SIZE = 0.02
    MAX_ASSETS = 25
    COMMISSION_RATE = 0.0005
    BID_ASK_SPREAD = 0.0003
    MARKET_IMPACT = 0.0002
    MIN_COMMISSION = 1.0
    RISK_FREE_RATE = 0.02
    SAFE_HAVEN_ASSETS = ["TLT", "GLD", "SHY", "IEF"]
    MAX_UNIVERSE_SIZE = 50


class DataManager:
    """
    Data acquisition and management for Yahoo Finance.

    CRITICAL: No synthetic data, no forward/backward filling.
    Only use data that actually exists.
    """

    def get_asset_universe(self, config: StrategyConfig) -> List[str]:
        """
        Define comprehensive asset universe.

        CRITICAL FIX: Removed duplicate 'XLRE'

        Returns:
            List of ticker symbols to download
        """
        universe = [
            "TLT",
            "GLD",
            "SHY",
            "IEF",
            "AGG",
            "SPY",
            "QQQ",
            "IWM",
            "DIA",
            "VTI",
            "VOO",
            "IVV",
            "XLF",
            "XLE",
            "XLK",
            "XLV",
            "XLI",
            "XLY",
            "XLP",
            "XLU",
            "XLRE",
            "XLB",
            "XLC",
            "EFA",
            "VEA",
            "EWJ",
            "EEM",
            "VWO",
            "FXI",
            "EWZ",
            "LQD",
            "HYG",
            "EMB",
            "AAPL",
            "MSFT",
            "GOOGL",
            "AMZN",
            "META",
            "NVDA",
            "TSLA",
            "JPM",
            "BAC",
            "JNJ",
            "PG",
            "WMT",
            "XOM",
            "CVX",
            "UNH",
            "V",
            "MA",
            "VNQ",
            "DBC",
            "USO",
            "VXX",
            "VIXY",
            config.BENCHMARK,
        ]
        universe = list(dict.fromkeys(universe))
        return universe[: config.MAX_UNIVERSE_SIZE]

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """
        Fetch historical OHLCV data from Yahoo Finance.

        CRITICAL: No forward-filling. Only return actual data points.

        Args:
            ticker: Ticker symbol
            start_date: Start date (YYYY-MM-DD)
            end_date: End date (YYYY-MM-DD)

        Returns:
            DataFrame with OHLCV data
        """
        buffer_start = (pd.to_datetime(start_date) - timedelta(days=150)).strftime("%Y-%m-%d")
        try:
            response_prices = {"historical": download_price_records(ticker, buffer_start, end_date)}
            data = response_prices
            if "historical" not in data or len(data["historical"]) < 100:
                return None
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            df.columns = [c.lower() for c in df.columns]
            required_cols = ["date", "open", "high", "low", "close", "volume"]
            if not all((col in df.columns for col in required_cols)):
                return None
            return df[required_cols]
        except Exception as e:
            return None

    def fetch_all_data(self, config: StrategyConfig) -> Dict[str, pd.DataFrame]:
        """
        Fetch all asset data.

        Returns:
            Dictionary mapping tickers to DataFrames
        """
        universe = self.get_asset_universe(config)
        print(f"\nFetching data for {len(universe)} assets...")
        print(f"Period: {config.START_DATE} to {config.END_DATE}")
        asset_data = {}
        successful = 0
        for ticker in universe:
            df = self.fetch_historical_data(ticker, config.START_DATE, config.END_DATE)
            if df is not None:
                asset_data[ticker] = df
                successful += 1
                if ticker in config.SAFE_HAVEN_ASSETS:
                    print(f"  {ticker}: {len(df)} days (SAFE HAVEN)")
                elif successful <= 5:
                    print(f"  {ticker}: {len(df)} days")
        print(f"\nSuccessfully downloaded: {successful}/{len(universe)} assets")
        safe_available = [a for a in config.SAFE_HAVEN_ASSETS if a in asset_data]
        if safe_available:
            print(f"Safe haven assets available: {', '.join(safe_available)}")
        else:
            print("WARNING: No safe haven assets available!")
        return asset_data


class HARSemivarianceCalculator:
    """
    Heterogeneous Autoregressive Semivariance model.

    METHODOLOGY:
    -----------
    HAR combines multiple time scales (daily, weekly, monthly) to capture
    volatility dynamics. Semivariance focuses on downside volatility only.

    Semivariance = Average of squared negative returns
    (Only counts returns below zero - captures downside risk)

    HAR model: Vol[t] = β_d*SV[t-1] + β_w*SV[t-5:t-1] + β_m*SV[t-22:t-1]
    where SV = realized semivariance

    ANTI-LOOK-AHEAD:
    ---------------
    - At time T, we use returns from T-1 backwards
    - Features are lagged to ensure T-1 → T flow
    - No future data contamination
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.daily_window = config.HAR_DAILY_WINDOW
        self.weekly_window = config.HAR_WEEKLY_WINDOW
        self.monthly_window = config.HAR_MONTHLY_WINDOW

    def calculate_realized_semivariance(self, returns: pd.Series, window: int) -> pd.Series:
        """
        Calculate realized semivariance over rolling window.

        Semivariance = Mean of squared negative returns
        Focuses on downside volatility (risk that matters to investors)

        CRITICAL: Uses ONLY negative returns (downside moves)

        Args:
            returns: Return series
            window: Rolling window size

        Returns:
            Realized semivariance series (annualized)
        """
        squared_negative_returns = np.where(returns < 0, returns**2, 0)
        min_periods_calc = min(max(1, window // 2), window)
        semivariance = (
            pd.Series(squared_negative_returns, index=returns.index)
            .rolling(window=window, min_periods=min_periods_calc)
            .mean()
        )
        semivariance_annual = semivariance * 252
        return semivariance_annual

    def calculate_har_features(self, returns: pd.Series) -> pd.DataFrame:
        """
        Calculate all HAR-Semivariance components.

        Components:
        - Daily: 1-day semivariance
        - Weekly: 5-day semivariance
        - Monthly: 22-day semivariance

        CRITICAL: Features are lagged to ensure T-1 → T flow

        Args:
            returns: Return series (must have enough history)

        Returns:
            DataFrame with lagged HAR components
        """
        daily_sv = self.calculate_realized_semivariance(returns, self.daily_window)
        weekly_sv = self.calculate_realized_semivariance(returns, self.weekly_window)
        monthly_sv = self.calculate_realized_semivariance(returns, self.monthly_window)
        har_features = pd.DataFrame(
            {"har_daily": daily_sv, "har_weekly": weekly_sv, "har_monthly": monthly_sv}
        )
        har_features["har_daily_lag"] = har_features["har_daily"].shift(1)
        har_features["har_weekly_lag"] = har_features["har_weekly"].shift(1)
        har_features["har_monthly_lag"] = har_features["har_monthly"].shift(1)
        har_features["har_composite"] = (
            0.5 * har_features["har_daily_lag"]
            + 0.3 * har_features["har_weekly_lag"]
            + 0.2 * har_features["har_monthly_lag"]
        )
        return har_features


@dataclass
class HHMMMapping:
    """Stores HHMM state mapping from training period."""

    state_to_regime: Dict[int, int]
    mean_volatilities: Dict[int, float]
    training_end_date: pd.Timestamp


class HierarchicalHMMDetector:
    """
    Hierarchical Hidden Markov Model for regime detection.

    ARCHITECTURE:
    ------------
    Uses HAR-Semivariance features + market features to detect 3 regimes:
    - State 0: Low Volatility (HAR < 33rd percentile)
    - State 1: Normal Volatility (33rd - 67th percentile)
    - State 2: High Volatility (> 67th percentile)

    CRITICAL FIX: Reduced from 4 to 3 states for better convergence
    - 4 states caused convergence issues
    - Only 2/4 regimes were being predicted
    - 3 states maps better to actual market dynamics

    WALK-FORWARD PROCESS:
    --------------------
    1. Calculate features using T-1 data (HAR + market indicators)
    2. Fit HMM on expanding historical window
    3. Map states to regimes based on historical volatility
    4. Freeze mapping and predict regime for T
    5. Refit quarterly (every 63 days)

    ANTI-LOOK-AHEAD GUARANTEE:
    -------------------------
    - All features use T-1 data
    - HMM trained only on past data
    - State mapping based on historical volatility only
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.n_states = config.HHMM_N_STATES
        self.current_mapping = None
        self.fold_history = []

    def build_hhmm_features(
        self,
        date: pd.Timestamp,
        har_features: pd.DataFrame,
        market_returns: pd.Series,
        market_volume: pd.Series,
        train_percentiles: Optional[Dict] = None,
    ) -> Optional[np.ndarray]:
        """
        Build feature vector for HHMM at time T using ONLY T-1 data.

        Features (all lagged):
        1. HAR composite semivariance (micro-volatility)
        2. HAR daily component
        3. HAR weekly component
        4. HAR monthly component
        5. Market return (20-day moving average)
        6. Market volume change (20-day)
        7. Return momentum (20-day cumulative return)

        CRITICAL FIX: Outlier clipping now uses training percentiles (not hard caps)

        Args:
            date: Current date T
            har_features: HAR feature DataFrame
            market_returns: Market return series
            market_volume: Market volume series
            train_percentiles: Optional dict with 99th percentile caps from training

        Returns:
            Feature vector or None if insufficient data
        """
        if date not in har_features.index:
            return None
        har_composite = har_features.loc[date, "har_composite"]
        har_daily = har_features.loc[date, "har_daily_lag"]
        har_weekly = har_features.loc[date, "har_weekly_lag"]
        har_monthly = har_features.loc[date, "har_monthly_lag"]
        if train_percentiles is not None:
            har_composite = np.clip(har_composite, 0, train_percentiles.get("har_composite", 0.5))
            har_daily = np.clip(har_daily, 0, train_percentiles.get("har_daily", 0.5))
            har_weekly = np.clip(har_weekly, 0, train_percentiles.get("har_weekly", 0.5))
            har_monthly = np.clip(har_monthly, 0, train_percentiles.get("har_monthly", 0.5))
        else:
            har_composite = np.clip(har_composite, 0, 0.5)
            har_daily = np.clip(har_daily, 0, 0.5)
            har_weekly = np.clip(har_weekly, 0, 0.5)
            har_monthly = np.clip(har_monthly, 0, 0.5)
        date_idx = market_returns.index.get_loc(date)
        if date_idx < 20:
            return None
        hist_returns = market_returns.iloc[date_idx - 20 : date_idx]
        market_ret_20d = hist_returns.mean() * 252
        hist_volume = market_volume.iloc[date_idx - 20 : date_idx]
        if len(hist_volume) > 0 and hist_volume.iloc[0] > 0:
            vol_change = hist_volume.iloc[-1] / hist_volume.iloc[0] - 1
        else:
            vol_change = 0
        momentum = (1 + hist_returns).prod() - 1
        if train_percentiles is not None:
            market_ret_20d = np.clip(
                market_ret_20d,
                train_percentiles.get("market_ret_min", -1.0),
                train_percentiles.get("market_ret_max", 1.0),
            )
            vol_change = np.clip(
                vol_change,
                train_percentiles.get("vol_change_min", -2.0),
                train_percentiles.get("vol_change_max", 2.0),
            )
            momentum = np.clip(
                momentum,
                train_percentiles.get("momentum_min", -0.5),
                train_percentiles.get("momentum_max", 0.5),
            )
        else:
            market_ret_20d = np.clip(market_ret_20d, -1.0, 1.0)
            vol_change = np.clip(vol_change, -2.0, 2.0)
            momentum = np.clip(momentum, -0.5, 0.5)
        features = np.array(
            [
                har_composite,
                har_daily,
                har_weekly,
                har_monthly,
                market_ret_20d,
                vol_change,
                momentum,
            ]
        )
        if np.any(np.isnan(features)) or np.any(np.isinf(features)):
            return None
        return features

    def compute_feature_percentiles(self, features_array: np.ndarray) -> Dict:
        """
        Compute 99th percentile caps from training features.

        CRITICAL FIX: Replaces hard caps with data-driven thresholds

        Args:
            features_array: Training feature array

        Returns:
            Dictionary with percentile thresholds
        """
        percentiles = {
            "har_composite": np.percentile(features_array[:, 0], 99),
            "har_daily": np.percentile(features_array[:, 1], 99),
            "har_weekly": np.percentile(features_array[:, 2], 99),
            "har_monthly": np.percentile(features_array[:, 3], 99),
            "market_ret_min": np.percentile(features_array[:, 4], 1),
            "market_ret_max": np.percentile(features_array[:, 4], 99),
            "vol_change_min": np.percentile(features_array[:, 5], 1),
            "vol_change_max": np.percentile(features_array[:, 5], 99),
            "momentum_min": np.percentile(features_array[:, 6], 1),
            "momentum_max": np.percentile(features_array[:, 6], 99),
        }
        return percentiles

    def fit_hhmm_expanding_window(
        self, features_array: np.ndarray, end_idx: int
    ) -> Tuple[hmm_lib.GaussianHMM, StandardScaler]:
        """
        Fit HHMM on expanding window [0:end_idx].

        CRITICAL FIX: Added verbose=False to suppress convergence warnings

        Args:
            features_array: Full feature array
            end_idx: Training end index (exclusive)

        Returns:
            Tuple of (fitted HMM model, fitted scaler)
        """
        X_train = features_array[:end_idx]
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        hhmm_model = hmm_lib.GaussianHMM(
            n_components=self.n_states,
            covariance_type=self.config.HHMM_COVARIANCE_TYPE,
            n_iter=self.config.HHMM_N_ITER,
            random_state=self.config.HHMM_RANDOM_STATE,
            tol=0.001,
            verbose=False,
        )
        hhmm_model.fit(X_train_scaled)
        return (hhmm_model, scaler)

    def create_regime_mapping(
        self, features_array: np.ndarray, states: np.ndarray, end_idx: int, end_date: pd.Timestamp
    ) -> HHMMMapping:
        """
        Map HMM states to volatility regimes based on historical volatility.

        CRITICAL: Uses ONLY training data to create mapping.

        CRITICAL FIX: Updated for 3 states (was 4)

        Mapping Logic:
        - Calculate average HAR composite for each state
        - Sort states by average volatility
        - Lowest vol → State 0 (Low)
        - Middle → State 1 (Normal)
        - Highest → State 2 (High)

        Args:
            features_array: Feature array
            states: Predicted states for training period
            end_idx: Training end index
            end_date: Training end date

        Returns:
            HHMMMapping object with state→regime mapping
        """
        state_volatilities = {}
        for state in range(self.n_states):
            mask = states[:end_idx] == state
            if mask.sum() > 0:
                avg_vol = features_array[:end_idx][mask, 0].mean()
                state_volatilities[state] = avg_vol
            else:
                state_volatilities[state] = 0
        sorted_states = sorted(state_volatilities.items(), key=lambda x: x[1])
        state_to_regime = {sorted_states[0][0]: 0, sorted_states[1][0]: 1, sorted_states[2][0]: 2}
        return HHMMMapping(
            state_to_regime=state_to_regime,
            mean_volatilities=state_volatilities,
            training_end_date=end_date,
        )

    def walk_forward_predict(
        self, har_features: pd.DataFrame, market_returns: pd.Series, market_volume: pd.Series
    ) -> Tuple[pd.DataFrame, List[Dict]]:
        """
        Execute walk-forward HHMM regime prediction.

        Returns:
            Tuple of (predictions DataFrame, fold history)
        """
        print("\n" + "=" * 80)
        print("HIERARCHICAL HMM REGIME DETECTION (WALK-FORWARD)")
        print("=" * 80)
        print("\nBuilding HHMM features...")
        all_features = []
        all_dates = []
        for date in har_features.index:
            if date in market_returns.index and date in market_volume.index:
                features = self.build_hhmm_features(
                    date, har_features, market_returns, market_volume
                )
                if features is not None:
                    all_features.append(features)
                    all_dates.append(date)
        if len(all_features) < self.config.MIN_TRAINING_DAYS:
            print(f"ERROR: Insufficient features: {len(all_features)}")
            raise ValueError("Insufficient data for HHMM")
        features_array = np.array(all_features)
        print(
            f"Features built: {len(features_array)} observations x {features_array.shape[1]} features"
        )
        results = pd.DataFrame({"date": all_dates, "regime": np.nan, "regime_name": None})
        results.set_index("date", inplace=True)
        min_train = self.config.MIN_TRAINING_DAYS
        refit_freq = self.config.REFIT_FREQUENCY
        print(f"\nWalk-forward configuration:")
        print(f"  Min training: {min_train} days")
        print(f"  Refit frequency: {refit_freq} days")
        last_refit_idx = min_train
        hhmm_model = None
        scaler = None
        mapping = None
        fold_num = 0
        regime_names = ["Low Vol", "Normal Vol", "High Vol"]
        for i in range(min_train, len(features_array)):
            days_since_refit = i - last_refit_idx
            if hhmm_model is None or days_since_refit >= refit_freq:
                fold_num += 1
                print(f"\nFold {fold_num}: Refitting HHMM at index {i}")
                print(f"  Training window: [0:{i}]")
                print(f"  Training dates: {all_dates[0].date()} to {all_dates[i - 1].date()}")
                hhmm_model, scaler = self.fit_hhmm_expanding_window(features_array, i)
                X_train_scaled = scaler.transform(features_array[:i])
                states_train = hhmm_model.predict(X_train_scaled)
                mapping = self.create_regime_mapping(
                    features_array, states_train, i, all_dates[i - 1]
                )
                print(f"  Regime mapping created:")
                for state, regime in mapping.state_to_regime.items():
                    vol = mapping.mean_volatilities[state]
                    print(f"    State {state} → {regime_names[regime]}: avg vol = {vol:.4f}")
                self.fold_history.append(
                    {
                        "fold": fold_num,
                        "train_end_idx": i,
                        "train_end_date": all_dates[i - 1],
                        "mapping": mapping.state_to_regime.copy(),
                        "mean_vols": mapping.mean_volatilities.copy(),
                    }
                )
                last_refit_idx = i
            X_current = features_array[i : i + 1]
            X_current_scaled = scaler.transform(X_current)
            state_pred = hhmm_model.predict(X_current_scaled)[0]
            regime_pred = mapping.state_to_regime[state_pred]
            current_date = all_dates[i]
            results.loc[current_date, "regime"] = regime_pred
            results.loc[current_date, "regime_name"] = regime_names[regime_pred]
        print(f"\n" + "=" * 80)
        print("HHMM DETECTION COMPLETE")
        print("=" * 80)
        print(f"Folds executed: {fold_num}")
        print(f"Predictions: {results['regime'].notna().sum()}")
        regime_dist = results["regime_name"].value_counts()
        print("\nRegime distribution:")
        for regime in regime_names:
            count = regime_dist.get(regime, 0)
            pct = count / len(results) * 100 if len(results) > 0 else 0
            print(f"  {regime:15s}: {count:4d} days ({pct:5.1f}%)")
        return (results, self.fold_history)


class MaxSharpeOptimizer:
    """
    Maximum Sharpe Ratio optimizer for Low Vol regime.

    Uses Ledoit-Wolf shrinkage for robust covariance estimation.
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.rf_rate = config.RISK_FREE_RATE

    def optimize(self, returns: pd.DataFrame, exposure: float) -> Dict[str, float]:
        """
        Optimize for maximum Sharpe ratio.

        Args:
            returns: Historical returns
            exposure: Target total exposure

        Returns:
            Dict of {asset: weight}
        """
        if len(returns) < 60:
            return {}
        mu = returns.mean() * 252
        try:
            lw = LedoitWolf()
            cov_shrunk, _ = (lw.fit(returns.values).covariance_, lw.shrinkage_)
            Sigma = pd.DataFrame(cov_shrunk * 252, index=returns.columns, columns=returns.columns)
        except:
            Sigma = returns.cov() * 252
        n = len(mu)

        def neg_sharpe(w):
            ret = np.dot(w, mu)
            vol = np.sqrt(np.dot(w.T, np.dot(Sigma, w)))
            return -(ret - self.rf_rate) / vol if vol > 0 else -np.inf

        constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - exposure}]
        bounds = [(0, self.config.MAX_POSITION_SIZE) for _ in range(n)]
        x0 = np.array([exposure / n] * n)
        try:
            result = minimize(
                neg_sharpe,
                x0,
                method="SLSQP",
                bounds=bounds,
                constraints=constraints,
                options={"maxiter": 1000, "ftol": 1e-09},
            )
            if result.success:
                weights = result.x
                weights = np.where(weights < self.config.MIN_POSITION_SIZE, 0, weights)
                if np.sum(weights) > 0:
                    weights = weights * exposure / np.sum(weights)
                    return {
                        asset: w
                        for asset, w in zip(mu.index, weights)
                        if w >= self.config.MIN_POSITION_SIZE
                    }
        except:
            pass
        return {}


class HRPOptimizer:
    """
    Hierarchical Risk Parity optimizer for Medium Vol regime.

    Lopez de Prado's HRP algorithm.
    """

    def __init__(self, config: StrategyConfig):
        self.config = config

    def optimize(self, returns: pd.DataFrame, exposure: float) -> Dict[str, float]:
        """
        Optimize using HRP.

        CRITICAL FIX: Drop NaN rows before correlation calculation

        Args:
            returns: Historical returns
            exposure: Target total exposure

        Returns:
            Dict of {asset: weight}
        """
        returns_clean = returns.dropna()
        if len(returns_clean) < 60:
            return {}
        try:
            corr = returns_clean.corr()
            if corr.isnull().any().any():
                valid_assets = corr.columns[~corr.isnull().any()]
                if len(valid_assets) < 3:
                    return {}
                returns_clean = returns_clean[valid_assets]
                corr = returns_clean.corr()
            dist = np.sqrt(0.5 * (1 - corr))
            condensed = squareform(dist.values)
            link = linkage(condensed, method="ward")
            sorted_idx = self._get_quasi_diagonal(link, len(corr))
            sorted_corr = corr.iloc[sorted_idx, sorted_idx]
            hrp_weights = self._recursive_bisection(sorted_corr)
            weight_dict = {}
            for i, asset in enumerate(corr.index):
                orig_idx = sorted_idx.index(i)
                weight = hrp_weights[orig_idx] * exposure
                if weight >= self.config.MIN_POSITION_SIZE:
                    weight_dict[asset] = weight
            return weight_dict
        except:
            return {}

    def _get_quasi_diagonal(self, link, n):
        """Get quasi-diagonal order from linkage."""
        sorted_idx = []

        def append_order(node_idx):
            if node_idx < n:
                sorted_idx.append(node_idx)
            else:
                left = int(link[node_idx - n][0])
                right = int(link[node_idx - n][1])
                append_order(left)
                append_order(right)

        append_order(len(link) + n - 1)
        return sorted_idx

    def _recursive_bisection(self, corr):
        """Recursive bisection for HRP."""
        weights = pd.Series(1.0, index=corr.index)
        clustered = [corr.index.tolist()]
        while len(clustered) > 0:
            clustered = [
                cluster[start:end]
                for cluster in clustered
                for start, end in ((0, len(cluster) // 2), (len(cluster) // 2, len(cluster)))
                if len(cluster) > 1
            ]
            for i in range(0, len(clustered), 2):
                if i + 1 < len(clustered):
                    left = clustered[i]
                    right = clustered[i + 1]
                    left_var = self._cluster_var(corr, left)
                    right_var = self._cluster_var(corr, right)
                    total_var = left_var + right_var
                    if total_var > 0:
                        left_w = right_var / total_var
                        right_w = left_var / total_var
                        for asset in left:
                            weights[asset] *= left_w
                        for asset in right:
                            weights[asset] *= right_w
        return weights.values

    def _cluster_var(self, corr, cluster):
        """Calculate cluster variance."""
        if len(cluster) == 1:
            return 1.0
        c_corr = corr.loc[cluster, cluster]
        w = np.ones(len(cluster)) / len(cluster)
        return np.dot(w.T, np.dot(c_corr.values, w))


class MSTHRPOptimizer:
    """
    Minimum Spanning Tree + HRP for High Vol regime.

    First clusters assets using MST, then applies HRP within each cluster.
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.hrp_optimizer = HRPOptimizer(config)

    def optimize(self, returns: pd.DataFrame, exposure: float) -> Dict[str, float]:
        """
        Optimize using MST clustering + HRP.

        Args:
            returns: Historical returns
            exposure: Target total exposure

        Returns:
            Dict of {asset: weight}
        """
        if len(returns) < 60:
            return {}
        try:
            corr = returns.corr().fillna(0)
            dist = np.sqrt(0.5 * (1 - corr))
            sparse_dist = csr_matrix(dist.values)
            mst = minimum_spanning_tree(sparse_dist)
            clusters = self._get_mst_clusters(mst.toarray(), corr.index)
            cluster_weights = {}
            for cluster_assets in clusters:
                if len(cluster_assets) >= 3:
                    cluster_returns = returns[cluster_assets]
                    cluster_weight_dict = self.hrp_optimizer.optimize(cluster_returns, 1.0)
                    cluster_weights.update(cluster_weight_dict)
            total = sum(cluster_weights.values())
            if total > 0:
                return {
                    asset: weight * exposure / total
                    for asset, weight in cluster_weights.items()
                    if weight * exposure / total >= self.config.MIN_POSITION_SIZE
                }
        except:
            pass
        return {}

    def _get_mst_clusters(self, mst_array, assets):
        """Extract clusters from MST."""
        n = len(assets)
        visited = set()
        clusters = []

        def dfs(node, cluster):
            visited.add(node)
            cluster.append(assets[node])
            for neighbor in range(n):
                if neighbor not in visited and mst_array[node, neighbor] > 0:
                    dfs(neighbor, cluster)

        for i in range(n):
            if i not in visited:
                cluster = []
                dfs(i, cluster)
                if len(cluster) >= 3:
                    clusters.append(cluster)
        return clusters if clusters else [[asset] for asset in assets[:5]]


class CVaRSafeHavenOptimizer:
    """
    CVaR + Safe Haven optimizer for Extreme Vol regime.

    Minimizes Conditional Value at Risk with bias toward safe haven assets.
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.confidence = config.CVAR_CONFIDENCE
        self.lookback = config.CVAR_LOOKBACK

    def optimize(
        self, returns: pd.DataFrame, exposure: float, asset_data: Dict
    ) -> Dict[str, float]:
        """
        Optimize using CVaR with safe haven bias.

        Args:
            returns: Historical returns
            exposure: Target exposure
            asset_data: Asset data dict (to check safe haven availability)

        Returns:
            Dict of {asset: weight}
        """
        if len(returns) < self.lookback:
            return {}
        safe_havens = [a for a in self.config.SAFE_HAVEN_ASSETS if a in returns.columns]
        if not safe_havens:
            return self._cvar_optimize(returns, exposure)
        safe_haven_weight = 0.7 * exposure
        risky_weight = 0.3 * exposure
        result = {}
        if safe_havens:
            per_safe_haven = safe_haven_weight / len(safe_havens)
            for asset in safe_havens:
                result[asset] = per_safe_haven
        risky_assets = [a for a in returns.columns if a not in safe_havens]
        if risky_assets:
            risky_returns = returns[risky_assets]
            risky_weights = self._cvar_optimize(risky_returns, risky_weight)
            result.update(risky_weights)
        return result

    def _cvar_optimize(self, returns: pd.DataFrame, exposure: float) -> Dict[str, float]:
        """
        Minimize CVaR.

        CVaR = Expected loss in worst (1-confidence)% cases
        """
        if len(returns) < 60:
            return {}
        n = len(returns.columns)
        T = len(returns)
        recent_returns = returns.iloc[-self.lookback :]

        def cvar_objective(w):
            port_returns = recent_returns.values @ w
            var_level = np.percentile(port_returns, (1 - self.confidence) * 100)
            losses = port_returns[port_returns <= var_level]
            cvar = -losses.mean() if len(losses) > 0 else 0
            return cvar

        constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - exposure}]
        bounds = [(0, self.config.MAX_POSITION_SIZE) for _ in range(n)]
        x0 = np.array([exposure / n] * n)
        try:
            result = minimize(
                cvar_objective,
                x0,
                method="SLSQP",
                bounds=bounds,
                constraints=constraints,
                options={"maxiter": 1000},
            )
            if result.success:
                weights = result.x
                weights = np.where(weights < self.config.MIN_POSITION_SIZE, 0, weights)
                if np.sum(weights) > 0:
                    weights = weights * exposure / np.sum(weights)
                    return {
                        asset: w
                        for asset, w in zip(returns.columns, weights)
                        if w >= self.config.MIN_POSITION_SIZE
                    }
        except:
            pass
        return {}


class BacktestEngine:
    """
    Comprehensive backtest engine with T-1 → T execution flow.

    CRITICAL FIXES APPLIED:
    - Calendar-aligned lookback windows (no positional indexing)
    - Proper handling of missing prices (no zero valuation)
    - Leverage financing costs when cash < 0
    - Data cleaning before optimization

    EXECUTION FLOW:
    --------------
    Day T-1 (Close):
      - Value portfolio at Close[T-1]
      - Calculate features from T-2 data
      - Detect regime
      - Optimize portfolio (using dates, not positions)
      - Generate signals (using Close[T-1] for sizing)

    Day T (Open):
      - Execute signals at Open[T] (if both Open[T] and Close[T] available)
      - Update positions

    Day T (Close):
      - Value portfolio at Close[T] (skip if missing)
      - Calculate returns: Close[T]/Close[T-1] - 1
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.trades = []
        self.leverage_spread = 0.02
        self.annual_borrow_rate = config.RISK_FREE_RATE + self.leverage_spread
        self.daily_borrow_rate = self.annual_borrow_rate / 252

    def calculate_transaction_cost(self, trade_value: float) -> float:
        """Calculate realistic transaction costs."""
        abs_value = abs(trade_value)
        commission = max(self.config.COMMISSION_RATE * abs_value, self.config.MIN_COMMISSION)
        spread = abs_value * self.config.BID_ASK_SPREAD
        impact = abs_value * self.config.MARKET_IMPACT
        return commission + spread + impact

    def run(
        self,
        regime_predictions: pd.DataFrame,
        asset_data: Dict[str, pd.DataFrame],
        har_calculator: HARSemivarianceCalculator,
    ) -> Dict:
        """
        Execute backtest with full walk-forward validation.

        CRITICAL FIXES:
        - Calendar-aligned lookback (not positional)
        - Skip assets without both Open[T] and Close[T]
        - Charge financing on negative cash
        - Clean NaN from optimization windows

        Returns:
            Comprehensive results dictionary
        """
        print("\n" + "=" * 80)
        print("BACKTEST EXECUTION")
        print("=" * 80)
        max_sharpe = MaxSharpeOptimizer(self.config)
        hrp = HRPOptimizer(self.config)
        cvar_safe = CVaRSafeHavenOptimizer(self.config)
        backtest_start = pd.to_datetime(self.config.START_DATE)
        backtest_end = pd.to_datetime(self.config.END_DATE)
        backtest_dates = regime_predictions[
            (regime_predictions.index >= backtest_start)
            & (regime_predictions.index <= backtest_end)
        ].copy()
        print(
            f"Backtest period: {backtest_dates.index[0].date()} to {backtest_dates.index[-1].date()}"
        )
        print(f"Trading days: {len(backtest_dates)}")
        print("\nPreparing returns matrix (calendar-aligned)...")
        available_assets = list(asset_data.keys())
        returns_dict = {}
        for asset in available_assets:
            df = asset_data[asset].copy()
            df = df.set_index("date").sort_index()
            df_aligned = df.reindex(backtest_dates.index)
            returns = df_aligned["close"].pct_change()
            if returns.notna().sum() / len(returns) >= 0.8:
                returns_dict[asset] = returns
        returns_matrix = pd.DataFrame(returns_dict)
        print(f"Assets with sufficient data: {len(returns_matrix.columns)}")
        capital = self.config.INITIAL_CAPITAL
        cash = capital
        positions = {asset: 0.0 for asset in returns_matrix.columns}
        portfolio_history = []
        last_rebalance = -self.config.REBALANCE_FREQUENCY
        total_financing_costs = 0.0
        print(f"\nExecuting backtest...")
        for day_idx, (date, row) in enumerate(backtest_dates.iterrows()):
            if day_idx % 50 == 0:
                print(f"  Day {day_idx + 1}/{len(backtest_dates)}: {date.strftime('%Y-%m-%d')}")
            regime = int(row["regime"]) if not pd.isna(row["regime"]) else 1
            regime_name = row["regime_name"] if row["regime_name"] else "Normal Vol"
            current_prices = {}
            for asset in returns_matrix.columns:
                if asset in asset_data:
                    df = asset_data[asset]
                    matches = df[df["date"].dt.date == date.date()]
                    if not matches.empty and pd.notna(matches.iloc[0]["close"]):
                        current_prices[asset] = matches.iloc[0]["close"]
            risky_value = 0.0
            for asset in returns_matrix.columns:
                if positions[asset] != 0:
                    if asset in current_prices:
                        risky_value += positions[asset] * current_prices[asset]
                    elif day_idx > 0 and len(portfolio_history) > 0:
                        prev_holding = portfolio_history[-1].get(f"holding_{asset}", 0)
                        risky_value += prev_holding
            if cash >= 0:
                cash += cash * (self.config.RISK_FREE_RATE / 252)
            else:
                daily_financing = abs(cash) * self.daily_borrow_rate
                cash -= daily_financing
                total_financing_costs += daily_financing
            portfolio_value = cash + risky_value
            days_since_rebal = day_idx - last_rebalance
            should_rebalance = days_since_rebal >= self.config.REBALANCE_FREQUENCY
            if should_rebalance and day_idx >= self.config.LOOKBACK_WINDOW:
                lookback_start = date - pd.Timedelta(days=int(self.config.LOOKBACK_WINDOW * 1.5))
                hist_returns = returns_matrix.loc[lookback_start:date]
                hist_returns_clean = hist_returns.dropna()
                if len(hist_returns_clean) >= 60:
                    exposure = self.config.REGIME_EXPOSURES[regime]
                    optimizer_name = self.config.REGIME_OPTIMIZERS[regime]
                    if optimizer_name == "MaxSharpe":
                        target_weights = max_sharpe.optimize(hist_returns_clean, exposure)
                    elif optimizer_name == "HRP":
                        target_weights = hrp.optimize(hist_returns_clean, exposure)
                    elif optimizer_name == "CVaR_SafeHaven":
                        target_weights = cvar_safe.optimize(
                            hist_returns_clean, exposure, asset_data
                        )
                    else:
                        target_weights = {}
                    if day_idx < len(backtest_dates) - 1:
                        next_date = backtest_dates.index[day_idx + 1]
                        exec_prices = {}
                        valid_for_trade = set()
                        for asset in returns_matrix.columns:
                            if asset in asset_data:
                                df = asset_data[asset]
                                matches = df[df["date"].dt.date == next_date.date()]
                                if not matches.empty:
                                    open_price = matches.iloc[0]["open"]
                                    close_price = matches.iloc[0]["close"]
                                    if pd.notna(open_price) and pd.notna(close_price):
                                        exec_prices[asset] = open_price
                                        valid_for_trade.add(asset)
                        trade_costs = 0.0
                        for asset, weight in target_weights.items():
                            if asset in valid_for_trade:
                                target_val = portfolio_value * weight
                                current_val = positions.get(asset, 0) * exec_prices[asset]
                                trade_val = target_val - current_val
                                if abs(trade_val) > portfolio_value * 0.005:
                                    cost = self.calculate_transaction_cost(trade_val)
                                    shares = trade_val / exec_prices[asset]
                                    positions[asset] = positions.get(asset, 0) + shares
                                    cash -= trade_val + cost
                                    trade_costs += cost
                        for asset in list(positions.keys()):
                            if asset not in target_weights and positions[asset] != 0:
                                if asset in valid_for_trade:
                                    value = positions[asset] * exec_prices[asset]
                                    cost = self.calculate_transaction_cost(abs(value))
                                    cash += value - cost
                                    positions[asset] = 0.0
                                    trade_costs += cost
                        if trade_costs > 0:
                            self.trades.append(
                                {"date": date, "regime": regime_name, "cost": trade_costs}
                            )
                        last_rebalance = day_idx
            history_entry = {
                "date": date,
                "value": portfolio_value,
                "cash": cash,
                "risky": risky_value,
                "regime": regime_name,
            }
            for asset in returns_matrix.columns:
                if asset in current_prices and positions[asset] != 0:
                    history_entry[f"holding_{asset}"] = positions[asset] * current_prices[asset]
            portfolio_history.append(history_entry)
        print("\nCalculating performance metrics...")
        portfolio_df = pd.DataFrame(portfolio_history)
        portfolio_df["return"] = portfolio_df["value"].pct_change()
        final_value = portfolio_df["value"].iloc[-1]
        total_return = final_value / self.config.INITIAL_CAPITAL - 1
        days = len(portfolio_df)
        annual_return = (1 + total_return) ** (252 / days) - 1 if days > 0 else 0
        returns = portfolio_df["return"].dropna()
        volatility = returns.std() * np.sqrt(252)
        sharpe = (annual_return - self.config.RISK_FREE_RATE) / volatility if volatility > 0 else 0
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min()
        results = {
            "portfolio_history": portfolio_df,
            "trade_history": self.trades,
            "metrics": {
                "initial_capital": self.config.INITIAL_CAPITAL,
                "final_value": final_value,
                "total_return": total_return,
                "annual_return": annual_return,
                "volatility": volatility,
                "sharpe_ratio": sharpe,
                "max_drawdown": max_dd,
                "num_trades": len(self.trades),
                "total_financing_costs": total_financing_costs,
            },
            "regime_predictions": backtest_dates,
            "asset_data": asset_data,
        }
        print("\n" + "=" * 80)
        print("BACKTEST COMPLETE")
        print("=" * 80)
        print(f"Final Value: ${final_value:,.0f}")
        print(f"Total Return: {total_return:.1%}")
        print(f"Annual Return: {annual_return:.1%}")
        print(f"Sharpe Ratio: {sharpe:.2f}")
        print(f"Max Drawdown: {max_dd:.1%}")
        print(f"Financing Costs: ${total_financing_costs:,.2f}")
        return results


class StrategyVisualizer:
    """
    Comprehensive visualization for HAR-HHMM-CVaR strategy.

    PLOTS:
    - Equity curve vs benchmark
    - Regime timeline with transitions
    - Drawdown analysis
    - Monthly returns heatmap
    - Rolling Sharpe ratio
    - Rolling beta (market sensitivity)
    - Exposure and leverage tracking
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.colors = {
            "strategy": "#2E86AB",
            "benchmark": "#A23B72",
            "positive": "#2ca02c",
            "negative": "#d62728",
            "regime_low": "#90EE90",
            "regime_normal": "#FFD700",
            "regime_high": "#FF6B6B",
        }
        self.regime_colors = {"Low Vol": "#90EE90", "Normal Vol": "#FFD700", "High Vol": "#FF6B6B"}

    def prepare_benchmark_data(
        self,
        portfolio_df: pd.DataFrame,
        asset_data: Dict[str, pd.DataFrame],
        initial_capital: float,
    ) -> pd.DataFrame:
        """
        Prepare benchmark returns aligned to strategy dates.

        Uses Close-to-Close returns matching strategy methodology.
        """
        if self.config.BENCHMARK not in asset_data:
            print(f"WARNING: Benchmark {self.config.BENCHMARK} not available")
            return None
        bench_df = asset_data[self.config.BENCHMARK].copy()
        bench_df["date"] = pd.to_datetime(bench_df["date"])
        benchmark_data = []
        for date in portfolio_df["date"].values:
            matches = bench_df[bench_df["date"].dt.date == pd.to_datetime(date).date()]
            if not matches.empty:
                benchmark_data.append(
                    {"date": pd.to_datetime(date), "close": matches.iloc[0]["close"]}
                )
            else:
                recent = bench_df[bench_df["date"] < pd.to_datetime(date)]
                if not recent.empty:
                    recent = recent.tail(1)
                    if (pd.to_datetime(date) - recent.iloc[0]["date"]).days <= 5:
                        benchmark_data.append(
                            {"date": pd.to_datetime(date), "close": recent.iloc[0]["close"]}
                        )
        if not benchmark_data:
            return None
        bench_aligned = pd.DataFrame(benchmark_data)
        bench_aligned["return"] = bench_aligned["close"].pct_change()
        first_close = bench_aligned["close"].iloc[0]
        bench_aligned["value"] = bench_aligned["close"] / first_close * initial_capital
        return bench_aligned

    def calculate_rolling_metrics(
        self, strategy_returns: pd.Series, benchmark_returns: pd.Series, window: int = 63
    ) -> pd.DataFrame:
        """
        Calculate rolling performance metrics.

        Metrics:
        - Rolling Sharpe ratio (risk-adjusted returns)
        - Rolling beta (market sensitivity)
        - Rolling volatility
        """
        metrics = pd.DataFrame(index=strategy_returns.index)
        rf_daily = self.config.RISK_FREE_RATE / 252
        rolling_mean = strategy_returns.rolling(window).mean() * 252
        rolling_std = strategy_returns.rolling(window).std() * np.sqrt(252)
        metrics["sharpe"] = (rolling_mean - self.config.RISK_FREE_RATE) / rolling_std
        rolling_cov = strategy_returns.rolling(window).cov(benchmark_returns)
        rolling_var = benchmark_returns.rolling(window).var()
        metrics["beta"] = rolling_cov / rolling_var
        metrics["volatility"] = rolling_std
        metrics = metrics.replace([np.inf, -np.inf], np.nan)
        return metrics

    def create_comprehensive_dashboard(
        self, results: Dict, initial_capital: float, save_path: Optional[str] = None
    ):
        """
        Generate comprehensive visualization dashboard.

        6-panel layout:
        1. Equity curve vs benchmark
        2. Regime timeline
        3. Drawdown analysis
        4. Monthly returns heatmap
        5. Rolling Sharpe ratio
        6. Rolling beta
        """
        portfolio_df = results["portfolio_history"]
        asset_data = results["asset_data"]
        regime_data = results["regime_predictions"]
        benchmark_df = self.prepare_benchmark_data(portfolio_df, asset_data, initial_capital)
        if benchmark_df is None:
            print(f"Cannot create benchmark comparison")
            benchmark_df = pd.DataFrame(
                {"date": portfolio_df["date"], "value": initial_capital, "return": 0.0}
            )
        fig = plt.figure(figsize=(20, 14))
        gs = fig.add_gridspec(4, 2, hspace=0.35, wspace=0.25)
        ax1 = fig.add_subplot(gs[0, :])
        ax1.plot(
            portfolio_df["date"],
            portfolio_df["value"],
            label=f"HAR-HHMM Strategy",
            color=self.colors["strategy"],
            linewidth=2.5,
            zorder=3,
        )
        ax1.plot(
            benchmark_df["date"],
            benchmark_df["value"],
            label=f"{self.config.BENCHMARK} Benchmark",
            color=self.colors["benchmark"],
            linewidth=2,
            alpha=0.8,
            zorder=2,
        )
        ax1.set_title(
            "Portfolio Performance: Strategy vs Benchmark", fontsize=14, fontweight="bold"
        )
        ax1.set_ylabel("Portfolio Value ($)", fontsize=11)
        ax1.legend(loc="upper left", fontsize=10)
        ax1.grid(True, alpha=0.3)
        ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
        final_strat = portfolio_df["value"].iloc[-1]
        final_bench = benchmark_df["value"].iloc[-1]
        strat_ret = (final_strat / initial_capital - 1) * 100
        bench_ret = (final_bench / initial_capital - 1) * 100
        textstr = f"Strategy: {strat_ret:+.1f}%\n{self.config.BENCHMARK}: {bench_ret:+.1f}%\nAlpha: {strat_ret - bench_ret:+.1f}%"
        ax1.text(
            0.98,
            0.02,
            textstr,
            transform=ax1.transAxes,
            fontsize=9,
            va="bottom",
            ha="right",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
        )
        ax2 = fig.add_subplot(gs[1, :])
        ax2_bg = ax2.twinx()
        ax2_bg.plot(
            portfolio_df["date"],
            (portfolio_df["value"] / initial_capital - 1) * 100,
            color="gray",
            alpha=0.2,
            linewidth=1,
        )
        ax2_bg.set_ylabel("Return (%)", fontsize=10, color="gray")
        ax2_bg.tick_params(axis="y", labelcolor="gray")
        regime_dates = regime_data.index
        regime_names = regime_data["regime_name"]
        for i in range(len(regime_dates) - 1):
            regime = regime_names.iloc[i]
            if pd.notna(regime) and regime in self.regime_colors:
                ax2.axvspan(
                    regime_dates[i],
                    regime_dates[i + 1],
                    alpha=0.3,
                    color=self.regime_colors[regime],
                )
        legend_patches = [
            mpatches.Patch(color=color, label=regime, alpha=0.5)
            for regime, color in self.regime_colors.items()
        ]
        ax2.legend(handles=legend_patches, loc="upper left", fontsize=9, ncol=3)
        ax2.set_title("Market Regime Detection Timeline (HHMM)", fontsize=14, fontweight="bold")
        ax2.set_ylabel("Regime", fontsize=11)
        ax2.set_yticks([])
        ax2.grid(True, alpha=0.3, axis="x")
        ax3 = fig.add_subplot(gs[2, 0])
        strategy_returns = portfolio_df["return"].dropna()
        bench_returns = benchmark_df["return"].dropna()
        portfolio_df_copy = portfolio_df.copy()
        portfolio_df_copy["cumulative"] = (1 + portfolio_df_copy["return"].fillna(0)).cumprod()
        portfolio_df_copy["running_max"] = portfolio_df_copy["cumulative"].expanding().max()
        portfolio_df_copy["drawdown"] = (
            (portfolio_df_copy["cumulative"] - portfolio_df_copy["running_max"])
            / portfolio_df_copy["running_max"]
            * 100
        )
        benchmark_df_copy = benchmark_df.copy()
        benchmark_df_copy["cumulative"] = (1 + benchmark_df_copy["return"].fillna(0)).cumprod()
        benchmark_df_copy["running_max"] = benchmark_df_copy["cumulative"].expanding().max()
        benchmark_df_copy["drawdown"] = (
            (benchmark_df_copy["cumulative"] - benchmark_df_copy["running_max"])
            / benchmark_df_copy["running_max"]
            * 100
        )
        ax3.fill_between(
            portfolio_df_copy["date"],
            portfolio_df_copy["drawdown"],
            0,
            alpha=0.6,
            color=self.colors["negative"],
            label="Strategy",
        )
        ax3.plot(
            benchmark_df_copy["date"],
            benchmark_df_copy["drawdown"],
            color=self.colors["benchmark"],
            linewidth=2,
            alpha=0.7,
            label=f"{self.config.BENCHMARK}",
        )
        ax3.set_title("Drawdown Analysis", fontsize=14, fontweight="bold")
        ax3.set_ylabel("Drawdown (%)", fontsize=11)
        ax3.legend(fontsize=9)
        ax3.grid(True, alpha=0.3)
        ax4 = fig.add_subplot(gs[2, 1])
        rolling_metrics = self.calculate_rolling_metrics(strategy_returns, bench_returns, window=63)
        dates_aligned = portfolio_df["date"].iloc[1:]
        ax4.plot(
            dates_aligned,
            rolling_metrics["sharpe"],
            color=self.colors["strategy"],
            linewidth=2,
            label="Strategy",
        )
        ax4.axhline(y=0, color="black", linestyle="-", linewidth=1, alpha=0.5)
        ax4.axhline(y=1, color="green", linestyle="--", linewidth=1, alpha=0.3, label="Good (>1.0)")
        ax4.axhline(
            y=2, color="darkgreen", linestyle="--", linewidth=1, alpha=0.3, label="Excellent (>2.0)"
        )
        ax4.set_title("Rolling Sharpe Ratio (63-day)", fontsize=14, fontweight="bold")
        ax4.set_ylabel("Sharpe Ratio", fontsize=11)
        ax4.legend(fontsize=9)
        ax4.grid(True, alpha=0.3)
        ax5 = fig.add_subplot(gs[3, 0])
        ax5.plot(dates_aligned, rolling_metrics["beta"], color=self.colors["strategy"], linewidth=2)
        ax5.axhline(
            y=1.0, color="black", linestyle="--", linewidth=1.5, alpha=0.7, label="Beta = 1.0"
        )
        ax5.axhline(y=0.0, color="gray", linestyle=":", linewidth=1, alpha=0.5)
        ax5.set_title(
            f"Rolling Beta vs {self.config.BENCHMARK} (63-day)", fontsize=14, fontweight="bold"
        )
        ax5.set_ylabel("Beta", fontsize=11)
        ax5.legend(fontsize=9)
        ax5.grid(True, alpha=0.3)
        ax6 = fig.add_subplot(gs[3, 1])
        portfolio_df_copy = portfolio_df.copy()
        portfolio_df_copy["date"] = pd.to_datetime(portfolio_df_copy["date"])
        portfolio_df_copy["year"] = portfolio_df_copy["date"].dt.year
        portfolio_df_copy["month"] = portfolio_df_copy["date"].dt.month
        monthly_returns = (
            portfolio_df_copy.groupby(["year", "month"])
            .apply(lambda x: (1 + x["return"]).prod() - 1)
            .reset_index(name="monthly_return")
        )
        heatmap_data = monthly_returns.pivot(index="year", columns="month", values="monthly_return")
        sns.heatmap(
            heatmap_data * 100,
            annot=True,
            fmt=".1f",
            cmap="RdYlGn",
            center=0,
            cbar_kws={"label": "Monthly Return (%)"},
            linewidths=0.5,
            linecolor="gray",
            ax=ax6,
        )
        month_names = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
        ax6.set_xticklabels([month_names[int(m) - 1] for m in heatmap_data.columns], rotation=0)
        ax6.set_title("Monthly Returns Heatmap", fontsize=14, fontweight="bold")
        ax6.set_xlabel("Month", fontsize=11)
        ax6.set_ylabel("Year", fontsize=11)
        fig.suptitle(
            "HAR-Semivariance + HHMM + CVaR Strategy Dashboard",
            fontsize=16,
            fontweight="bold",
            y=0.995,
        )
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            print(f"\nDashboard saved: {save_path}")
        plt.show()

    def create_regime_analysis(self, results: Dict, save_path: Optional[str] = None):
        """
        Detailed regime analysis visualization.

        Shows regime performance statistics and transitions.
        """
        portfolio_df = results["portfolio_history"]
        regime_data = results["regime_predictions"]
        fig, axes = plt.subplots(2, 1, figsize=(16, 10))
        regime_counts = regime_data["regime_name"].value_counts()
        colors_list = [self.regime_colors[r] for r in regime_counts.index]
        bars = axes[0].bar(regime_counts.index, regime_counts.values, color=colors_list, alpha=0.7)
        total = regime_counts.sum()
        for bar, count in zip(bars, regime_counts.values):
            height = bar.get_height()
            pct = count / total * 100
            axes[0].text(
                bar.get_x() + bar.get_width() / 2.0,
                height,
                f"{count}\n({pct:.1f}%)",
                ha="center",
                va="bottom",
                fontsize=10,
            )
        axes[0].set_title("Regime Distribution", fontsize=14, fontweight="bold")
        axes[0].set_ylabel("Number of Days", fontsize=11)
        axes[0].grid(True, alpha=0.3, axis="y")
        portfolio_with_regime = portfolio_df.copy()
        portfolio_with_regime["regime"] = None
        for idx, row in portfolio_with_regime.iterrows():
            date = row["date"]
            if date in regime_data.index:
                portfolio_with_regime.loc[idx, "regime"] = regime_data.loc[date, "regime_name"]
        regime_returns = {}
        for regime in self.regime_colors.keys():
            mask = portfolio_with_regime["regime"] == regime
            returns = portfolio_with_regime.loc[mask, "return"].dropna()
            if len(returns) > 0:
                regime_returns[regime] = returns.values
        bp = axes[1].boxplot(
            [regime_returns.get(r, [0]) for r in self.regime_colors.keys()],
            labels=list(self.regime_colors.keys()),
            patch_artist=True,
            showfliers=False,
        )
        for patch, regime in zip(bp["boxes"], self.regime_colors.keys()):
            patch.set_facecolor(self.regime_colors[regime])
            patch.set_alpha(0.7)
        axes[1].axhline(y=0, color="black", linestyle="--", alpha=0.5)
        axes[1].set_title("Daily Returns Distribution by Regime", fontsize=14, fontweight="bold")
        axes[1].set_ylabel("Daily Returns", fontsize=11)
        axes[1].grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches="tight")
            print(f"Regime analysis saved: {save_path}")
        plt.show()


def main():
    """Execute HAR-Semivariance + HHMM + CVaR strategy."""
    print("=" * 80)
    print("HAR-SEMIVARIANCE + HIERARCHICAL HMM + CVaR STRATEGY")
    print("=" * 80)
    print("\nARCHITECTURE:")
    print("  Level 1 (Macro): HHMM determines total risk exposure")
    print("  Level 2 (Micro): Regime-specific optimizer adjusts weights")
    print("\nCRITICAL FIX: Reduced to 3 regimes for better convergence")
    print("  - 4 states caused convergence issues")
    print("  - Only 2/4 regimes were being predicted")
    print("  - 3 states maps better to market dynamics")
    print("\nOPTIMIZERS BY REGIME:")
    print("  Low Vol (120%):    Max Sharpe")
    print("  Normal Vol (100%): HRP")
    print("  High Vol (50%):    CVaR + Safe Haven")
    print("=" * 80)
    config = StrategyConfig()
    print("\n1. DATA ACQUISITION")
    print("-" * 80)
    data_manager = DataManager()
    asset_data = data_manager.fetch_all_data(config)
    if len(asset_data) < 10:
        raise ValueError(f"Insufficient assets: {len(asset_data)}")
    print("\n2. HAR-SEMIVARIANCE CALCULATION")
    print("-" * 80)
    if "SPY" not in asset_data:
        raise ValueError("SPY data required for HAR calculation")
    spy_df = asset_data["SPY"].copy()
    spy_df = spy_df.set_index("date").sort_index()
    spy_returns = spy_df["close"].pct_change()
    har_calculator = HARSemivarianceCalculator(config)
    har_features = har_calculator.calculate_har_features(spy_returns)
    print(f"HAR features calculated: {len(har_features)} observations")
    print("\n3. HIERARCHICAL HMM REGIME DETECTION")
    print("-" * 80)
    hhmm_detector = HierarchicalHMMDetector(config)
    regime_predictions, fold_history = hhmm_detector.walk_forward_predict(
        har_features, spy_returns, spy_df["volume"]
    )
    print("\n4. BACKTEST EXECUTION")
    print("-" * 80)
    backtest_engine = BacktestEngine(config)
    results = backtest_engine.run(regime_predictions, asset_data, har_calculator)
    print("\n5. GENERATING VISUALIZATIONS")
    print("-" * 80)
    visualizer = StrategyVisualizer(config)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    visualizer.create_comprehensive_dashboard(
        results, config.INITIAL_CAPITAL, save_path=f"har_hhmm_dashboard_{timestamp}.png"
    )
    visualizer.create_regime_analysis(
        results, save_path=f"har_hhmm_regime_analysis_{timestamp}.png"
    )
    print("\n" + "=" * 80)
    print("EXECUTION COMPLETE")
    print("=" * 80)
    return (config, results)


if __name__ == "__main__":
    config, results = main()
